# Filter drifters trajectories


In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from rasterio.transform import Affine

import pynsitu as pyn

_______
# Select under swath drifters - create and store dataset

In [2]:
drifters_sources = 'all_med_variational_10min_v0.nc'
drifter_file = os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv'))
df = pd.read_csv(drifter_file)

/var/folders/fn/z858c2qj1lz65xr0z5mdvbf40000gp/T/ipykernel_37299/205826707.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(drifter_file)


_______

In [3]:
ds = xr.open_dataset('/Users/mdemol/DATA_DRIFTERS/drifters_harmonized_old/L2/all_med_variational_10min_v0.nc')
df['drifter_id'] = df.drifter_id.astype(str)
ds = ds.sel(drifter_id = df.drifter_id.unique())

In [6]:
ds

<xarray.Dataset> Size: 759MB
Dimensions:             (drifter_id: 156, datetime: 43406)
Coordinates:
  * datetime            (datetime) datetime64[ns] 347kB 2023-03-27T12:50:00 ....
  * drifter_id          (drifter_id) <U15 9kB '300534060315840' ... '4694474'
Data variables: (12/18)
    x                   (drifter_id, datetime) float64 54MB ...
    y                   (drifter_id, datetime) float64 54MB ...
    drifter_type        (drifter_id) <U7 4kB ...
    cruise_id           (drifter_id) <U9 6kB ...
    lonc                (drifter_id) float64 1kB ...
    latc                (drifter_id) float64 1kB ...
    ...                  ...
    acceleration_north  (drifter_id, datetime) float64 54MB ...
    acceleration        (drifter_id, datetime) float64 54MB ...
    X                   (drifter_id, datetime) float64 54MB ...
    gap_mask            (drifter_id, datetime) float64 54MB ...
    gaps                (drifter_id, datetime) float64 54MB ...
    sign_acceleration   (drifter_id, datetime) float64 54MB ...
Attributes:
    description:                     L2 product - applied smoothing method on...
    smoothing_method:                Variational
    smoothing_method_param_dict:     {'acc_cut': 0.0003, 'position_error': 70...
    interpolation_sampling:          10min
    version:                         v0
    geographical_projection:         Asimuthal Equidistant projection centere...
    geographical_projection_pyproj:  pyproj.Proj(proj="aeqd", lat_0=latc, lon...
    citation_str:                    
    distribution_str:                These data were collected and made freel...
    generation_date:                 2024-09-30 08:37:54.325171

In [9]:
def filter_traj(ds_drifter, df_coloc, T=12, cutoff = 2.5):
    """Return dataset with filtered trajectories acceleration and coriolis term
    T: window length days
    cutoff : cut off frequency in cpd
    """
    from scipy.signal import filtfilt
    from scipy.integrate import cumulative_trapezoid
    from scipy.optimize import minimize
    
    #df_ = df.iloc[0]
    #ds_ = ds.sel(drifter_id = str(df_.drifter_id))[['velocity_east', 'velocity_north']]
    
    # coefficients
    dt = (ds.datetime.diff("datetime") / pd.Timedelta("1D")).mean().values  # in days
    from pynsitu.tseries import generate_filter

    dss = ds[["velocity_east", "velocity_north", 'latitude']].fillna(0)
    
    taps = generate_filter(band="low", dt=dt, T=T, bandwidth=cutoff)
    print(len(taps))
    dss['velocity_east'] = xr.DataArray(
        filtfilt(taps, 1, dss.velocity_east),
        dims=["drifter_id", "datetime"],
        )
    dss['velocity_north'] = xr.DataArray(
        filtfilt(taps, 1, dss.velocity_north),
        dims=["drifter_id","datetime"],
        )
    #except:
    #    print("padlen modified")
    #    vx = filtfilt(taps, 1, dss.velocity_east, padlen=len(dss.velocity_east))
    #    vy = filtfilt(taps, 1, dss.velocity_north, padlen=len(dss.velocity_east))

    cutoffstr = str(cutoff).replace(".", "")

    dss['acceleration_east'] = (
        (dss.velocity_east.differentiate("datetime", datetime_unit = '1s'))
        .assign_attrs(**ds.acceleration_east.attrs)
        .assign_attrs(
            description=ds.acceleration_east.attrs["description"]
            + f" filtered with {cutoff} cpd frequency",
            cutoff=cutoff,
        )
    )
    dss['acceleration_north'] = (
        (dss.velocity_north.differentiate("datetime", datetime_unit = '1s'))
        .assign_attrs(**ds.acceleration_north.attrs)
        .assign_attrs(
            description=ds.acceleration_north.attrs["description"]
            + f" filtered with {cutoff} cpd frequency",
            cutoff=cutoff,
        )
    )
    dff = dss.to_dataframe()
    dffc = df.set_index(['drifter_id', 'datetime'])
    dff = dff.loc[dffc.index].reset_index()
    dff['row_number'] = np.arange(len(dff))
    dff = dff.set_index('row_number')
    assert np.all(dff.datetime == df.set_index('row_number').datetime)# check dataframe are aligned
    add = df.set_index('row_number')[[ 'longitude', 'latitude', 'pass_number','time_to_swot','cycle_number','cycle_date','drifter_type']]
    dh = pd.concat([dff, add], axis=1)
    return dh
    #dh.to_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', f"_filtered{str(cutoff).replace('.','')}.csv")))

In [10]:
dff = filter_traj(ds, df, T=12, cutoff = 2.5)

ValueError: The cutoff argument must be at most one-dimensional.

In [11]:
from sstats import tseries as ts

def spectrum_ds(ds, nperseg="20D", detrend=False):
    ds = df.reset_index().set_index(["time", "id"]).to_xarray()
    vc = [("x", "y"), ("u", "v"), ("ax", "ay")]
    E = []
    for tup in vc:
        ds_ = ds.ts.spectrum(unit="1D", nperseg=nperseg, detrend=detrend, complex=tup)
        E.append(ds_)
    return xr.merge(E)


ModuleNotFoundError: No module named 'sstats'